# Initialize Git Repository
https://docs.ycrc.yale.edu/clusters-at-yale/guides/github/

In [ ]:
#git init -b main  # Initializes the repo with 'main' as the default branch
#git add .
#git commit -m "Initial commit"
#git remote add origin git@github.com:innacohen/mod-extract.git
#git push -u origin main --force

# Download mod files as a single JSON file

In [1]:
import os
import pandas as pd
import requests
import json
import re
from tqdm import tqdm

# Load dataset from Excel
df = pd.read_excel("../data/model_db_annotations.xlsx")

# Output JSON files
output_json = "../data/mod_files.json"
failed_json = "../data/failed_downloads.json"

# List to store mod file data and failed downloads
mod_files_data = []
failed_downloads = []

# Function to convert ModelDB URL to direct download URL
def get_direct_download_url(url):
    match = re.search(r"https://modeldb\.science/(\d+)\?tab=2&file=(.+)", url)
    if match:
        model_id, file_path = match.groups()
        return f"https://modeldb.science/getModelFile?model={model_id}&file={file_path}", file_path
    return None, None  # Return None if the URL doesn't match the expected pattern

# Function to fetch .mod file content and store in JSON
def fetch_mod_file(row_id, file_hash, raw_sha, count, url):
    direct_url, file_path = get_direct_download_url(url)
    
    # Default structure for JSON entry
    file_entry = {
        "row_id": row_id,
        "file_hash": file_hash,
        "raw_sha": raw_sha,
        "count": count,
        "url": url,
        "download_url": direct_url,
        "content": None  # Default to None (indicating missing content)
    }

    if not direct_url:
        print(f"🚨 Skipping invalid URL: {url}")
        failed_downloads.append(file_entry)
        mod_files_data.append(file_entry)  # Store even failed ones
        return

    try:
        response = requests.get(direct_url, timeout=10)
        response.raise_for_status()
        file_entry["content"] = response.text  # Store the downloaded content
    except requests.exceptions.RequestException as e:
        print(f"❌ Failed to fetch {direct_url}: {e}")
        failed_downloads.append(file_entry)  # Log failed download

    mod_files_data.append(file_entry)  # Store the file entry (success or failure)

# Select rows from 467 (corresponding to row_id 468) and take 11 rows
start = 467
n_rows = 20 
df_subset = df.iloc[start:start+n_rows]

# Process selected files
for _, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc="Fetching .mod files"):
    fetch_mod_file(
        row["row_id"], 
        row["file_hash"], 
        row["raw_sha"], 
        row["count"], 
        row["url"]
    )

# Save all .mod data to JSON (including failed downloads)
with open(output_json, "w", encoding="utf-8") as json_file:
    json.dump(mod_files_data, json_file, indent=4)

# Save failed downloads separately for easy retrying
if failed_downloads:
    with open(failed_json, "w", encoding="utf-8") as json_file:
        json.dump(failed_downloads, json_file, indent=4)
    print(f"⚠️ Some downloads failed. Check {failed_json} for details.")

print(f"\n✅ All .mod files (including failures) saved in {output_json}")


/home/imc33/.conda/envs/nn/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
Fetching .mod files:  15%|█▌        | 3/20 [00:00<00:01, 10.72it/s]

❌ Failed to fetch https://modeldb.science/getModelFile?model=58173&file=b05oct26/na.mod: 403 Client Error: Forbidden for url: https://modeldb.science/getModelFile?model=58173&file=b05oct26/na.mod


Fetching .mod files: 100%|██████████| 20/20 [00:01<00:00, 11.60it/s]

⚠️ Some downloads failed. Check ../data/failed_downloads.json for details.

✅ All .mod files (including failures) saved in ../data/mod_files.json


# Read JSON file

In [2]:
import os
import pandas as pd
import requests
import json
import re
from bs4 import BeautifulSoup
from tqdm import tqdm

In [3]:
def View(df, rows=5, cols=None, width=None):
    """Displays the first `rows` of the DataFrame like R's View() by adjusting Pandas settings."""
    
    # Show only the first `rows` of the DataFrame
    with pd.option_context(
        "display.max_rows", rows,  # Limit number of rows shown
        "display.max_columns", cols,  # Show all columns
        "display.max_colwidth", width,  # Show full column width
        "display.expand_frame_repr", False  # Prevent column wrapping
    ):
        display(df.head(rows))  # Show only the first `rows`


In [82]:
# Function to extract mod directory from the URL
def extract_mod_dir(url):
    match = re.search(r"file=([^/]+)/[^/]+\.mod", url)  # Extract the directory name before the .mod file
    return match.group(1) if match else None  # Return directory name if found, else None

# Function to extract mod file name without extension
def extract_mod_name(url):
    match = re.search(r"/([^/]+)\.mod$", url)  # Get filename without extension
    return match.group(1) if match else None  # Return only the name (e.g., 'na')

# Function to extract model_id from the URL
def extract_model_id(url):
    match = re.search(r"https://modeldb\.science/(\d+)", url)
    return int(match.group(1)) if match else None  # Convert to integer

# Function to determine exclusion reason with snake_case labels
def determine_exclusion(row):
    if pd.isna(row["content"]):
        return "not_found"  # Standardized exclusion for missing files
    if "x86_64" in row["url"]:
        return "x86_64"  # Standardized exclusion for architecture issues
    return None  # Keep valid entries as None (not excluded)

# Function to extract all TITLE occurrences from .mod content
def extract_title(content):
    if pd.isna(content):  # If content is None or NaN, return None
        return None
    matches = re.findall(r"^TITLE\s+(.+)", content, re.MULTILINE)  # Find all TITLE lines
    return matches if matches else None  # Return list if multiple, else None

# Function to extract all COMMENT sections from .mod content
def extract_comment(content):
    if pd.isna(content):  # If content is None or NaN, return None
        return None
    matches = re.findall(r"COMMENT\s+(.*?)\s+ENDCOMMENT", content, re.DOTALL)  # Capture all COMMENT sections
    return matches if matches else None  # Return list if multiple, else None

# Function to extract all SUFFIX occurrences from .mod content
def extract_mod_suffix(content):
    if pd.isna(content):  # If content is None or NaN, return None
        return None
    matches = re.findall(r"SUFFIX\s+(\w+)", content)  # Find all SUFFIX statements
    return matches if matches else None  # Return list if multiple, else None

# Function to extract all USEION occurrences from .mod content
def extract_mod_use_ion(content):
    if pd.isna(content):  # If content is None or NaN, return None
        return None
    matches = re.findall(r"USEION\s+(\w+)", content)  # Find all USEION statements
    return matches if matches else None  # Return list if multiple, else None



# Function to extract all ions listed after READ but stopping before WRITE
def extract_mod_read_ion(content):
    if pd.isna(content):  # If content is None or NaN, return None
        return None
    
    matches = re.findall(r"USEION\s+\w+\s+READ\s+([\w,\s]+?)(?:\s+WRITE|\n|$)", content)  # Capture all READ ions
    read_ions = [ion.strip() for match in matches for ion in re.split(r"[,\s]+", match) if ion]  # Flatten & clean list
    
    return read_ions if read_ions else None  # Return list if found, else None


# Function to extract RANGE variables (before GLOBAL)
def extract_mod_range(content):
    if pd.isna(content):
        return None

    # Find RANGE block and stop at the next GLOBAL block if present
    match = re.search(r"RANGE\s+([\w\s,]+?)(?=\s+GLOBAL|\s*$)", content, re.MULTILINE)

    if not match:
        return None

    # Extract variables from the RANGE block
    range_vars = [var.strip() for var in match.group(1).split(",") if var.strip()]

    return range_vars if range_vars else None  # Return cleaned list

# Function to extract all GLOBAL variables while removing "GLOBAL" itself
def extract_mod_global(content):
    if pd.isna(content):
        return None

    # Find all GLOBAL blocks, allowing for spaces and newlines
    matches = re.findall(r"GLOBAL\s+([\w\s,]+)", content, re.MULTILINE)

    if not matches:
        return None

    # Join all GLOBAL blocks into a single string, replacing newlines with spaces
    cleaned_text = " ".join(matches).replace("\n", " ")

    # Split on commas and whitespace, ensuring variables are correctly separated
    global_vars = [var.strip() for var in re.split(r"[,\s]+", cleaned_text) if var and var.upper() != "GLOBAL"]

    return global_vars if global_vars else None  # Return cleaned list


# Function to extract parameter names and values as a dictionary
def extract_mod_parameters(content):
    if pd.isna(content):  # If content is None or NaN, return None
        return None
    
    # Regex to find parameter names and values, ignoring units in parentheses
    matches = re.findall(r"(\w+)\s*=\s*([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)", content)  
    
    # Convert to dictionary (float conversion only for valid numbers)
    param_dict = {name: float(value) for name, value in matches if re.match(r"^-?\d*\.?\d+(?:[eE][-+]?\d+)?$", value)}
    
    return param_dict if param_dict else None  # Return dictionary or None if empty

# Function to extract webpage heading
def extract_heading(url):
    try:
        response = requests.get(url, timeout=10)  # Fetch the webpage
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        
        # Try extracting heading from the most relevant tag
        heading = soup.find("h1")  # Look for <h1> (main title)
        return heading.text.strip() if heading else None  # Return text or None
    except requests.exceptions.RequestException:
        return None  # Return None if the request fail

# Function to extract citation (text inside parentheses)
def extract_citation(heading):
    if pd.isna(heading):
        return None
    match = re.search(r"\(([^)]+)\)", heading)  # Find text inside parentheses
    return match.group(1) if match else None  # Extract citation


# Function to extract first author(s) (removes "et al." and "al" correctly)
def extract_first_author(citation):
    if pd.isna(citation):
        return None

    # Extract first author(s) before "et al" or variants
    match = re.search(r"^([\w\s&\-,]+?)(?:\s+et\s+al\.?|et)?(?:,|\s|$)", citation)  
    first_author = match.group(1).strip() if match else None  

    # Remove any trailing "al" left behind
    if first_author:
        first_author = re.sub(r"\b(al)\b", "", first_author, flags=re.IGNORECASE).strip()

    return first_author

# Function to extract the first year from citation (including shortened years like '20)
def extract_year(citation):
    if pd.isna(citation):
        return None
    match = re.search(r"\b(19|20)?\d{2}\b|'\d{2}", citation)  # Find 4-digit or short year ('20)
    if match:
        return match.group(0).replace("'", "")  # Remove apostrophe but keep year as '20' if short
    return None  # Return None if no year found

In [83]:
# Load JSON into DataFrame
df = pd.read_json("../data/mod_files.json")

# Set "row_id" as the index
df.set_index("row_id", inplace=True)

# Exclude mod files
df["mod_exclude"] = df.apply(determine_exclusion, axis=1)  # Apply exclusion function

# Extract features from url
df["mod_heading"] = df["url"].apply(extract_heading)  # Extract heading from webpage
df["mod_citation"] = df["mod_heading"].apply(extract_citation)
df["mod_first_author"] = df["mod_citation"].apply(extract_first_author)
df["mod_year"] = df["mod_citation"].apply(extract_year)  # Now handles multiple years


df["mod_dir"] = df["url"].apply(extract_mod_dir)
df["mod_name"] = df["url"].apply(extract_mod_name)
df["mod_model_id"] = df["url"].apply(extract_model_id)

#  Extract features from content
df["mod_title"] = df["content"].apply(extract_title)
df["mod_comment"] = df["content"].apply(extract_comment)
df["mod_suffix"] = df["content"].apply(extract_mod_suffix)
df["mod_use_ion"] = df["content"].apply(extract_mod_use_ion)
df["mod_read_ion"] = df["content"].apply(extract_mod_read_ion)
df["mod_range"] = df["content"].apply(extract_mod_range)
df["mod_global"] = df["content"].apply(extract_mod_global)
df["mod_parameters"] = df["content"].apply(extract_mod_parameters)

In [63]:
df.columns

Index(['file_hash', 'raw_sha', 'count', 'url', 'download_url', 'content',
       'mod_exclude', 'mod_heading', 'mod_citation', 'mod_first_author',
       'mod_year', 'mod_dir', 'mod_name', 'mod_model_id', 'mod_title',
       'mod_comment', 'mod_suffix', 'mod_use_ion', 'mod_read_ion', 'mod_range',
       'mod_global', 'mod_parameters'],
      dtype='object')

In [84]:
View(df)

,file_hash,raw_sha,count,url,download_url,content,mod_exclude,mod_heading,mod_citation,mod_first_author,mod_year,mod_dir,mod_name,mod_model_id,mod_title,mod_comment,mod_suffix,mod_use_ion,mod_read_ion,mod_range,mod_global,mod_parameters
row_id,,,,,,,,,,,,,,,,,,,,,,
468,25b50b5441b9a5492122214e0648e0529ec055f2d6e97b63969da7a3b4ade149,f81e972ce9d730e924c8280e95395b26346853b9c0b397821437b68475ef9607,3,https://modeldb.science/58173?tab=2&file=b05oct26/na.mod,https://modeldb.science/getModelFile?model=58173&file=b05oct26/na.mod,None,not_found,None,None,None,None,b05oct26,na,58173,None,None,None,None,None,None,None,None
469,3fb039ac47d8004f4d9da9bc3da43bf89ce3e9c632e87d48f55a569276610e81,5a432e65a6d72ef8943f0ae6e833100108f341f793c0a092f3b8ae497273a04c,1,https://modeldb.science/116983?tab=2&file=theta/h.mod,https://modeldb.science/getModelFile?model=116983&file=theta/h.mod,"TITLE I-h channel from Magee 1998 for distal dendrites\r\n\r\nUNITS {\r\n\t(mA) = (milliamp)\r\n\t(mV) = (millivolt)\r\n\r\n}\r\n\r\nPARAMETER {\r\n\tv \t\t(mV)\r\n ehd \t\t(mV) \r\n\tcelsius \t(degC)\r\n\tghdbar=.0001 \t(mho/cm2)\r\n vhalfl=-100 \t(mV)\r\n\tkl=-12\r\n vhalft=-75 \t(mV)\r\n a0ta=0.001 \t(/ms)\r\n a0td=0.009 \t(/ms)\r\n zetat=2.2 \t(1)\r\n gmt=.4 \t(1)\r\n\tq10=4.5\r\n\tqtl=1\r\n}\r\n\r\n\r\nNEURON {\r\n\tSUFFIX hd\r\n\tNONSPECIFIC_CURRENT i\r\n RANGE ghdbar, vhalfl, tau\r\n GLOBAL linf,taua, taud\r\n}\r\n\r\nSTATE {\r\n l\r\n}\r\n\r\nASSIGNED {\r\n\ti (mA/cm2)\r\n linf \r\n\ttau\r\n taua\r\n taud\r\n ghd\r\n}\r\n\r\nINITIAL {\r\n\trate(v)\r\n\tl=linf\r\n}\r\n\r\n\r\nBREAKPOINT {\r\n\tSOLVE states METHOD cnexp\r\n\tghd = ghdbar*l\r\n\ti = ghd*(v-ehd)\r\n\r\n}\r\n\r\n\r\nFUNCTION alpt(v(mV)) {\r\n alpt = exp(0.0378*zetat*(v-vhalft)) \r\n}\r\n\r\nFUNCTION bett(v(mV)) {\r\n bett = exp(0.0378*zetat*gmt*(v-vhalft)) \r\n}\r\n\r\nDERIVATIVE states { : exact when v held constant; integrates over dt step\r\n rate(v)\r\n if (l<linf) {tau=taua} else {tau=taud}\r\n l' = (linf - l)/tau\r\n}\r\n\r\nPROCEDURE rate(v (mV)) { :callable from hoc\r\n LOCAL a,qt\r\n qt=q10^((celsius-33)/10)\r\n a = alpt(v)\r\n linf = 1/(1 + exp(-(v-vhalfl)/kl))\r\n taua = bett(v)/(qtl*qt*a0ta*(1+a))\r\n taud = bett(v)/(qtl*qt*a0td*(1+a))\r\n}\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n",None,CA1 pyramidal neuron: h channel-dependent deficit of theta oscill. resonance (Marcelin et al. 2008),Marcelin et al. 2008,Marcelin,2008,theta,h,116983,[I-h channel from Magee 1998 for distal dendrites\r],None,[hd],None,None,"[ghdbar, vhalfl, tau]","[linf, taua, taud]","{'ghdbar': 0.0001, 'vhalfl': -100.0, 'kl': -12.0, 'vhalft': -75.0, 'a0ta': 0.001, 'a0td': 0.009, 'zetat': 2.2, 'gmt': 0.4, 'q10': 4.5, 'qtl': 1.0, 'linf': 1.0}"
470,338c3525e3a21e4cdc628b3224a949883cf12b2765f872278fc0859841a3bb8e,e8daa7ec610c947cbfa1bef33e16ba2748e74da7e2596e60d04a7150f509f2d5,2,https://modeldb.science/136026?tab=2&file=djurisic2008/na.mod,https://modeldb.science/getModelFile?model=136026&file=djurisic2008/na.mod,"\nCOMMENT\n\nna.mod\n\nSodium channel, Hodgkin-Huxley style kinetics. \n\n\nqi is not well constrained by the data, since there are no points\nbetween -80 and -55. So this was fixed at 5 while the thi1,thi2,Rg,Rd\nwere optimized using a simplex least square proc\n\nvoltage dependencies are shifted approximately +5mV from the best\nfit to give higher threshold\n\nuse with kd.mod\n\nAuthor: Upinder S. Bhalla, California Institute of Technology\nJ. of Neurophysiology, V69, N6, 1993\n\nENDCOMMENT\n\nINDEPENDENT {t FROM 0 TO 1 WITH 1 (ms)}\n\nNEURON {\n\tSUFFIX na\n\tUSEION na READ ena WRITE ina\n\tRANGE m, h, gna, gbar, vshift\n\tGLOBAL thm1, thm2, qm1, qm2, thi1, thi2, qi, qinf, thinf\n\tGLOBAL minf, hinf, mtau, htau, ina\n\tGLOBAL Am1, Am2, Rd, Rg\n\tGLOBAL q10, temp, tadj, vmin, vmax\n}\n\nPARAMETER {\n\tgbar = 258.272 \t(pS/um2)\t: 0.12 mho/cm2\n\tvshift = 0\t(mV)\t\t: voltage shift (affects all)\n\t\t\t\t\t\t\t\t\n\tthm1 = -70.3833\t(mV)\t\t: v 1/2 for act\t\t(-42)\n\tthm2 = -21.8432\t(mV)\

In [71]:
View(df.loc[:,["mod_range"]], 10)

,mod_range
row_id,
468,None
469,"[ghdbar, vhalfl, tau\r\n GLOBAL linf, taua, taud]"
470,"[m, h, gna, gbar, vshift\n\tGLOBAL thm1, thm2, qm1, qm2, thi1, thi2, qi, qinf, thinf\n\tGLOBAL minf, hinf, mtau, htau, ina\n\tGLOBAL Am1, Am2, Rd, Rg\n\tGLOBAL q10, temp, tadj, vmin, vmax]"
471,"[onset, tau1, tau2, Mgetha, gamma, gmax, e, i\n\tNONSPECIFIC_CURRENT i]"
472,"[gkmbar, ik, g\r\n\r\n\r\n\tRANGE M_inf, tau_M, M]"
473,None
474,"[gbar\n USEION k READ ek WRITE ik\n RANGE Vkd, ik, vrev\n RANGE a1, a2, b1, b2]"
475,None
476,"[r, i \n NONSPECIFIC_CURRENT i]"


In [33]:
! git add .
! git commit -m "fixed mod range and global functions t"
! git push 

[main fabfac1] extract mod range and global parameters
 1 file changed, 336 insertions(+), 28 deletions(-)
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 36 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (4/4), 12.64 KiB | 6.32 MiB/s, done.
Total 4 (delta 0), reused 0 (delta 0), pack-reused 0
To github.com:innacohen/mod-extract.git
   650383c..fabfac1  main -> main


In [ ]:
!pwd

In [ ]:
View(df,3)

In [28]:
sample = """ 
TITLE K-Cl cotransporter KCC2

INDEPENDENT {t FROM 0 TO 1 WITH 10 (ms)}

NEURON {
	SUFFIX kcc2
	USEION k READ ko, ki WRITE ik
	USEION cl READ clo, cli WRITE icl VALENCE -1
	RANGE ik, icl
	GLOBAL U, S, Vi
}

UNITS {
	(mV)	= (millivolt)
	(molar) = (1/liter)
	(mM)	= (millimolar)
	(um)	= (micron)
	(mA)	= (milliamp)
	(mol)	= (1)
	FARADAY	= 96485.309 (coul/mole)
	PI	= (pi) (1)
}

PARAMETER {
	S = 5654.87 (um2)
  	Vi = 8913.48 (um3)
	U = 0.0003    (mM/ms)
}

ASSIGNED {
	ik		(mA/cm2)
	icl		(mA/cm2)
	ko		(mM)
	ki		(mM)
  	clo   (mM)
  	cli   (mM)
}

BREAKPOINT {
	LOCAL rate
	rate = pumprate(ki,ko,cli,clo)
	ik =  rate
	icl = -rate
}

FUNCTION pumprate(ki,ko,cli,clo) {
	pumprate = U*log((ki*cli)/(ko*clo))*(FARADAY*Vi/(S*1e4))
}
"""

In [77]:
sample2 = """
NEURON {
	SUFFIX na
	USEION na READ ena WRITE ina
	RANGE m, h, gna, gbar, vshift
	GLOBAL thm1, thm2, qm1, qm2, thi1, thi2, qi, qinf, thinf
	GLOBAL minf, hinf, mtau, htau, ina
	GLOBAL Am1, Am2, Rd, Rg
	GLOBAL q10, temp, tadj, vmin, vmax
}

"""

In [80]:
import re
import pandas as pd



In [81]:
extract_mod_range(sample2)

['m', 'h', 'gna', 'gbar', 'vshift']

In [ ]:
! git push